# Know your data

### What the agent gave us, and what it could not have known

We just watched an agent download a dataset, analyse it, and produce a chart in
about ninety seconds. The chart is real, the code runs, and the number on it is
arithmetically correct.

This notebook is where we find out what it means.

We will go through three layers, and each one needs more from us than the last:

| | layer | the question | who can answer it |
|---|---|---|---|
| 1 | **Provenance** | *Which file is this, actually?* | anyone who looks |
| 2 | **Statistics** | *Which average is this?* | someone who knows statistics |
| 3 | **Domain** | *What is missing, and what does the average hide?* | **someone who knows the subject** |

The agent could have caught layer 1 and 2 if we had asked precisely enough.
It could not have caught layer 3 at all, the information required is not in
the file.

---
## 0 · Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.width", 120)
pd.set_option("display.max_rows", 80)

# Validated categorical palette (colour-blind safe).
BLUE, ORANGE, AQUA, YELLOW, MAGENTA = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"
INK, MUTED, GRID, FAINT = "#0b0b0b", "#52514e", "#e6e5e1", "#d6d5d1"

plt.rcParams.update({
    "figure.figsize": (10, 5.5),
    "figure.dpi": 110,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.titleweight": "normal",
    "axes.labelcolor": MUTED,
    "axes.edgecolor": GRID,
    "axes.linewidth": 1.0,
    "text.color": INK,
    "xtick.color": MUTED,
    "ytick.color": MUTED,
    "legend.frameon": False,
})


def style(ax, title=None, subtitle=None, ylabel=None):
    """Recessive grid, no top/right spines, optional subtitle under the title."""
    ax.grid(axis="y", color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    if title:
        # The subtitle sits at y=1.02 and grows upward, so the title has to be
        # padded clear of it - one line's worth of pad per subtitle line.
        n_lines = subtitle.count("\n") + 1 if subtitle else 0
        ax.set_title(title, loc="left",
                     pad=(18 + 14 * (n_lines - 1)) if subtitle else 10)
    if subtitle:
        ax.text(0, 1.02, subtitle, transform=ax.transAxes,
                fontsize=10, color=MUTED, va="bottom")
    if ylabel:
        ax.set_ylabel(ylabel)
    return ax

---
## 1 · What did the agent actually download?

Before anything else. Not *"is the analysis right"* but **"what is this file."**

Point `CSV` at whatever landed in the starter folder during the demo.

In [ ]:
CANDIDATES = [
    Path("data/gapminder.csv"),      # what the agent downloaded, live
    Path("../fallback/gapminder.csv"),          # pre-downloaded safety net
]

CSV = next((p for p in CANDIDATES if p.exists()), None)
if CSV is None:
    raise FileNotFoundError(f"No data found. Looked in: {[str(p) for p in CANDIDATES]}")

df = pd.read_csv(CSV)
print(f"loaded: {CSV}")
df.head()

In [ ]:
print("rows      :", len(df))
print("columns   :", list(df.columns))
print("countries :", df["country"].nunique())
print("years     :", df["year"].unique().tolist())

### The one diagnostic worth memorising

A dataset like this is a **panel**: countries measured repeatedly over time.
The question that decides whether *any* year-on-year comparison is meaningful is
whether the panel is **balanced**. Does every year measure the same countries?

If the composition changes between years, then a change in the average may be
telling you about *who got measured*, not about the world.

In [ ]:
per_year = df.groupby("year")["country"].nunique()
print(per_year.to_string())
print()
print("balanced panel?", per_year.nunique() == 1)

> **If that printed `True`**: good, this is the clean 142-country file. Keep going;
> section 2 shows what the *other* file would have done to us.
>
> **If it printed `False`**: the agent fetched `gapminder_unfiltered.csv`. Section 2
> is now live rather than hypothetical.

---
## 2 · The other six files on that same page

Remember the search at the start? Typing `gapminder` into that repository returns
**seven** files:

| file | rows | countries | years | what it would have done to us |
|---|---:|---:|---|---|
| `gapminderDataFiveYear.csv` | 1704 | 142 | 1952–2007 | ← the one we wanted |
| `gapminderDataFiveYear2.csv` | 1704 | 142 | 1952–2007 | same data, extra column, harmless |
| `gapminder_with_codes.csv` | 1704 | 142 | 1952–2007 | same data, ISO codes, harmless |
| `gapminder-unclean.csv` | 1704 | 142 | 1952–2007 | **missing values**, silently skipped by `.mean()` |
| `gapminder_unfiltered.csv` | 3313 | 187 | 1950–2007 | different countries in different years |
| `gapminder2007.csv` | 142 | 142 | **none** | no time dimension, no trend possible |
| `gapminderData_Americas_2007.csv` | 25 | 25 | 2007 only | the Americas, one year |

We asked for "the Gapminder dataset". **Nothing in that phrase distinguishes any
of them**, and three of the seven would have quietly answered a different question.

Take the fifth one, the closest in name to what we got, and apply the agent's
exact analysis to it.

In [ ]:
ALT_LOCAL = Path("../fallback/gapminder_unfiltered.csv")
ALT_URL = "https://raw.githubusercontent.com/plotly/datasets/master/gapminder_unfiltered.csv"

alt = pd.read_csv(ALT_LOCAL if ALT_LOCAL.exists() else ALT_URL)

alt_per_year = alt.groupby("year")["country"].nunique()
print("countries measured per year, unfiltered file:")
print(alt_per_year.head(16).to_string())

Twenty-four countries in 1953, one hundred and forty-four in 1952. The
five-yearly "census" years have everybody; the years in between have only the
handful of rich countries that kept continuous records.

Now apply the agent's exact analysis, mean life expectancy per year, to it.

In [ ]:
naive_alt = alt.groupby("year")["lifeExp"].mean()

fig, ax = plt.subplots()
ax.plot(naive_alt.index, naive_alt.values, color=BLUE, linewidth=2, marker="o",
        markersize=4, markerfacecolor="white", markeredgewidth=1.4)
style(ax,
      title="The same analysis, applied to the other file",
      subtitle="Mean life expectancy per year · gapminder_unfiltered.csv",
      ylabel="Years")
ax.set_xlabel("Year")
plt.tight_layout()
plt.show()

The world did not oscillate between 50 and 68 years of life expectancy every
twelve months. **The sawtooth is the country list changing**, not life expectancy
changing. In the sparse years we are averaging Switzerland, Sweden and Japan; in
the census years we are averaging the whole world.

This chart is what "the same, correct code" produces on a file whose name differs
by one word.

---
## 3 · Which line produced the headline number?

Back to the agent's own script.

In [ ]:
# ---- PASTE ZONE -------------------------------------------------------
# Paste the line from the agent's analysis.py that computes the number.
#
# Pre-filled with what qwen3-coder actually produced. This exact line
# came back in 10 out of 10 benchmark runs.

global_life_expectancy = df.groupby('year')['lifeExp'].mean()

# -----------------------------------------------------------------------
naive = global_life_expectancy
print(f"headline number, {naive.index.max()}: {naive.iloc[-1]:.2f} years")

One line.
```python
global_life_expectancy = df.groupby('year')['lifeExp'].mean()
```

**What is that the average of?**

It is the average of **142 numbers** — one per country. Not the average of six
billion people. Those are different quantities and they answer different
questions.

The name says **global**. The chart title says **Global Average Life Expectancy
Over Time**. The word appears twice, and neither time it is earned. What was
computed is a mean over 142 country values, with Iceland and China contributing
equally.

Nothing here is a bug. The arithmetic is correct, the code is clean, the chart is
honest about being a chart. **The label is doing the misleading, and the label is
the only part anybody reads.**

---
## 4 · Layer 2: one country, one vote

An unweighted mean treats every row as one observation of equal importance.
Here a row is a *country*. So:

In [ ]:
latest = df[df["year"] == df["year"].max()]
latest[latest["country"].isin(["Iceland", "China", "India", "Luxembourg"])][
    ["country", "pop", "lifeExp"]
].sort_values("pop")

Iceland has roughly **three hundred thousand** people. China has **one point three
billion**. In the agent's average they count the same.

If we want "the life expectancy of the average *person*", we have to weight each
country by how many people live in it.

In [ ]:
tmp = df.assign(_pl=df["lifeExp"] * df["pop"])
agg = tmp.groupby("year").agg(
    unweighted=("lifeExp", "mean"),
    _pl=("_pl", "sum"),
    _pop=("pop", "sum"),
)
comp = pd.DataFrame({
    "unweighted (per country)": agg["unweighted"],
    "weighted (per person)": agg["_pl"] / agg["_pop"],
})
comp["gap"] = comp["unweighted (per country)"] - comp["weighted (per person)"]
comp.round(2)

In [ ]:
fig, ax = plt.subplots()
ax.plot(comp.index, comp["unweighted (per country)"], color=BLUE, linewidth=2,
        marker="o", markersize=4, markerfacecolor="white", markeredgewidth=1.4,
        label="Average country")
ax.plot(comp.index, comp["weighted (per person)"], color=ORANGE, linewidth=2,
        marker="o", markersize=4, markerfacecolor="white", markeredgewidth=1.4,
        label="Average person")

last = comp.index.max()
# Push each end-label away from the other: the lower line's label goes down,
# the upper line's goes up, or they collide (the lines are <2 years apart).
for col, colour, dy in [("unweighted (per country)", BLUE, -16),
                        ("weighted (per person)", ORANGE, 8)]:
    ax.annotate(f"{comp.loc[last, col]:.1f}",
                xy=(last, comp.loc[last, col]),
                xytext=(8, dy), textcoords="offset points",
                color=colour, fontweight="bold")

style(ax,
      title="Two defensible averages, two different answers",
      subtitle=f"Gap in {last}: {comp.loc[last, 'gap']:+.2f} years",
      ylabel="Life expectancy (years)")
ax.set_xlabel("Year")
ax.set_xlim(comp.index.min() - 1, last + 5)   # room for the end-labels
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

Roughly two years apart. Neither line is a bug, they are answers to two
different questions, and **only one of them is the question we asked**.

This is the layer a statistician catches. It is also the layer a *better model*
would probably catch, if we had phrased the request more carefully.

Now for the layer no model can catch for us.

---
## 5 · Layer 3a: who is not in this file?

Every analysis so far has said "the world". Let us check who the world contains.

In [ ]:
present = set(df["country"])

check = ["Russia", "Ukraine", "Kazakhstan", "Uzbekistan", "Belarus", "Georgia",
         "Armenia", "Azerbaijan", "Estonia", "Latvia", "Lithuania", "Moldova",
         "Luxembourg", "Qatar", "United Arab Emirates", "Papua New Guinea"]

missing = [c for c in check if c not in present]
print(f"absent from this dataset ({len(missing)}):")
for c in missing:
    print("   ", c)

In [ ]:
covered = df.loc[df["year"] == 2007, "pop"].sum()
world_2007 = 6.6e9  # UN estimate
print(f"population covered in 2007 : {covered / 1e9:.2f} billion")
print(f"actual world population    : {world_2007 / 1e9:.2f} billion")
print(f"coverage                   : {covered / world_2007:.0%}")

Ninety-five percent coverage sounds harmless but **look at which five percent.**

The absent countries are, almost exactly, the post-Soviet states. That is not a
random five percent, it is the one large population in this period whose life
expectancy went sharply **down**. Russian male life expectancy fell by roughly
six years between 1990 and 1994.

So the dataset that the agent used to tell us the world is getting healthier has
had the largest counter-example of the era quietly removed from it. Not
maliciously, Gapminder's five-year file just requires a complete 1952–2007
series, and the USSR did not exist for most of it.

**Nothing in the CSV says this.** There is no column for it, no README shipped
with the download, no warning in the agent's output. You know it or you don't.

---
## 5b · Layer 3b: what does a global average erase?

The agent's conclusion was that the world is getting healthier, and the line goes
up, so that reads as true. Break the same data apart by continent.

In [ ]:
by_cont = df.groupby(["continent", "year"])["lifeExp"].mean().unstack(0)
by_cont.round(1)

In [ ]:
COLOURS = {"Africa": ORANGE, "Americas": BLUE, "Asia": AQUA,
           "Europe": MAGENTA, "Oceania": YELLOW}

fig, ax = plt.subplots(figsize=(10.5, 5.8))
ax.axvspan(1987, 2002, color=ORANGE, alpha=0.08, zorder=0)

for cont in by_cont.columns:
    s = by_cont[cont]
    ax.plot(s.index, s.values, color=COLOURS[cont], linewidth=2.4)
    # direct end-labels: identity is never colour-alone
    ax.annotate(cont, xy=(s.index.max(), s.iloc[-1]), xytext=(8, -4),
                textcoords="offset points", color=COLOURS[cont],
                fontweight="normal", fontsize=10.5)

ax.annotate("Africa: flat for 15 years", xy=(1994, 53.5), xytext=(1968, 44),
            color=ORANGE, fontsize=10.5,
            arrowprops=dict(arrowstyle="->", color=ORANGE, linewidth=1.4))

style(ax,
      title="The global average is a lid on top of this",
      subtitle="Mean life expectancy by continent, unweighted within continent",
      ylabel="Life expectancy (years)")
ax.set_xlabel("Year")
ax.set_xlim(df["year"].min() - 1, df["year"].max() + 11)
plt.tight_layout()
plt.show()

In [ ]:
africa = df[df["continent"] == "Africa"].groupby("year")["lifeExp"].mean()
print("Africa, mean life expectancy:")
print(africa.loc[1982:2007].round(2).to_string())
print()
print(f"change 1987 -> 2002: {africa.loc[2002] - africa.loc[1987]:+.2f} years")

Fifteen years, and the number does not move. That is the HIV/AIDS epidemic, and
it is completely invisible in the global line, which rose smoothly straight
through it.

At country level it is not a plateau at all, it is a collapse.

In [ ]:
HIGHLIGHT = {"Zimbabwe": BLUE, "Botswana": ORANGE, "Rwanda": AQUA, "Cambodia": MAGENTA}

fig, ax = plt.subplots(figsize=(10.5, 5.8))

for country, grp in df.groupby("country"):
    if country not in HIGHLIGHT:
        ax.plot(grp["year"], grp["lifeExp"], color=FAINT, linewidth=0.8, zorder=1)

for country, colour in HIGHLIGHT.items():
    grp = df[df["country"] == country]
    ax.plot(grp["year"], grp["lifeExp"], color=colour, linewidth=2.4, zorder=3)
    ax.annotate(country, xy=(grp["year"].max(), grp["lifeExp"].iloc[-1]),
                xytext=(8, -4), textcoords="offset points",
                color=colour, fontweight="normal", fontsize=10.5)

style(ax,
      title="Every country in the dataset, four of them named",
      subtitle="Rwanda 1992: 23.6 years · Zimbabwe 1987–2002: 62.4 → 40.0",
      ylabel="Life expectancy (years)")
ax.set_xlabel("Year")
ax.set_xlim(df["year"].min() - 1, df["year"].max() + 12)
plt.tight_layout()
plt.show()

In [ ]:
df[df["country"].isin(HIGHLIGHT)].pivot_table(
    index="year", columns="country", values="lifeExp"
).round(1)

Rwanda in 1992 reads **23.6 years**. That is the genocide, showing up as a single
number in a spreadsheet.

The agent's one-line conclusion, *"life expectancy is rising, the world is
getting healthier"*, is not false. It is just the least informative true
sentence available about this data.

---
## 6 · Layer 3c: when does this data stop?

In [ ]:
print("last year in the dataset :", df["year"].max())
print("current year             :", pd.Timestamp.today().year)
print("gap                      :", pd.Timestamp.today().year - int(df["year"].max()), "years")

The agent wrote its conclusion in the present tense. The data stops before the
2008 financial crisis, before the Syrian war, before Ebola, before COVID-19,
which reduced life expectancy in many countries by more than a year.

The chart says "the world is getting healthier". It is describing 2007.

---
## 7 · The honest chart

Everything we now know, stated on the chart itself.

In [ ]:
world = comp["weighted (per person)"]

fig, ax = plt.subplots(figsize=(10.5, 6))

for cont in by_cont.columns:
    s = by_cont[cont]
    colour = ORANGE if cont == "Africa" else FAINT
    width = 2.2 if cont == "Africa" else 1.4
    ax.plot(s.index, s.values, color=colour, linewidth=width, zorder=2)
    ax.annotate(cont, xy=(s.index.max(), s.iloc[-1]), xytext=(8, -4),
                textcoords="offset points", fontsize=10,
                color=ORANGE if cont == "Africa" else MUTED,
                fontweight="normal" if cont == "Africa" else "normal")

ax.plot(world.index, world.values, color=BLUE, linewidth=3, zorder=4)
ax.annotate("World\n(per person)", xy=(world.index.max(), world.iloc[-1]),
            xytext=(8, -14), textcoords="offset points",
            color=BLUE, fontweight="bold", fontsize=10.5)

style(ax,
      title="Life expectancy of the average person, 1952–2007",
      subtitle=("Population-weighted · 142 countries, ~95% of world population · "
                "excludes Russia and all post-Soviet states\n"
                "Data: Gapminder, five-year series. Ends 2007."),
      ylabel="Life expectancy (years)")
ax.set_xlabel("Year")
ax.set_xlim(df["year"].min() - 1, df["year"].max() + 9)
plt.tight_layout()
plt.savefig("honest_life_expectancy.png", dpi=200, bbox_inches="tight")
plt.show()

print(f"The average person's life expectancy rose from {world.iloc[0]:.1f} years "
      f"in {world.index.min()} to {world.iloc[-1]:.1f} in {world.index.max()},")
print("but Africa's average did not move between 1987 and 2002, and the countries")
print("whose life expectancy fell in the 1990s are not in this dataset.")

---
## 8 · What actually went wrong

Nothing here was a hallucination. The agent did not invent a number, crash, or
write broken code. Every line it produced was correct.

| | what happened | could the agent have known? |
|---|---|---|
| **Provenance** | fetched one of two near-identically named files | only if we had said which |
| **Statistics** | averaged countries, not people | only if we had said "per person" |
| **Missing data** | called 142 countries "the world" | **no**, nothing in the file says who is absent |
| **Hidden structure** | reported a rising global average | **no**, it is not wrong, just uninformative |
| **Vintage** | wrote about 2007 in the present tense | only if we had told it today's date matters |

The first two are prompting problems. **The last three are not.** No amount of
prompt engineering, and no larger model, recovers the fact that Russia is absent
from this file, because that fact is not in the file.

That is the part of the job that is still yours.

### The checklist

Before an agent's number goes on your slide:

1. **What is this file?** Rows, columns, units, date range, and *who is missing*.
2. **Is the panel balanced?** `df.groupby(time)[unit].nunique()`, one line.
3. **What is the average the average of?** Rows? People? Countries? Say the unit out loud.
4. **Which single line produced the headline number?** If you cannot point at it, you cannot defend it.
5. **Disaggregate once.** By region, by group, by anything. If the subgroups disagree with the total, the total is hiding something.
6. **When does the data stop?** Compare to today's date before writing a present-tense sentence.

None of these take more than a minute. All of them require you to be the one who
knows what the data is *about*.